# Lattice Effects and FBS Grading on Electron Mixing Fractions (N_k=1)

Refines qDP/hDP fractions by modeling:
- Discrete hyperedge walks (12-neighbor branching per vertex)
- Graded FBS broadcast strength (~ r^{-2} falloff)
Goal: tighter bound on electron δμ_e

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Golden ratio and base gradients
phi = (1 + np.sqrt(5)) / 2
base_gradient = 1.0
gradients = {
    'eDP': (phi - 1) * base_gradient,  # favorable
    'hDP': base_gradient,
    'qDP': phi * base_gradient         # least favorable
}

N_k = 1.0
T = N_k
n_samples = 500000  # balanced for speed/precision

# Simulate discrete hyperedge walks (12 neighbors per vertex)
# Each step: random walk with branching factor 12, distance-weighted
steps = np.random.geometric(p=1/12, size=n_samples)  # geometric dist for path length
r_norm = steps / 12.0  # normalize to mean path ~1

# FBS grading: strength falls as r^{-2}
fbs_grade = 1 / (r_norm**2 + 0.1)  # avoid div by zero
fbs_grade /= fbs_grade.max()  # normalize [0,1]

# Modulated noise (lattice + FBS)
noise_scale = 0.05 * fbs_grade
noise = np.random.normal(0, noise_scale[:, np.newaxis], (n_samples, 3))

energies = np.array([gradients['eDP'], gradients['hDP'], gradients['qDP']]) + noise

# Boltzmann probabilities
probs = np.exp(-energies / T)
probs /= probs.sum(axis=1)[:, np.newaxis]

# Average fractions
avg_fracs = probs.mean(axis=0)
std_fracs = probs.std(axis=0)
labels = ['eDP', 'hDP', 'qDP']

print("Lattice + FBS-refined average fractions (electron, N_k=1):")
for label, avg, std in zip(labels, avg_fracs, std_fracs):
    print(f"{label}: {avg:.6f} ± {std:.6f}")

# Plot qDP distribution
plt.hist(probs[:, 2], bins=120, density=True, alpha=0.7, color='orange', label='qDP')
plt.axvline(avg_fracs[2], color='red', linestyle='--', label=f'Mean qDP = {avg_fracs[2]:.6f}')
plt.xlabel('qDP Fraction')
plt.ylabel('Density')
plt.title('Lattice + FBS Effects on qDP Distribution (N_k=1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../figures/electron-qdp-lattice-fbs.png')
plt.show()

## Preliminary g-2 Estimate with Lattice + FBS Effects

In [ ]:
C = 9.6e-7
alpha = 1 / 137.035999084
S = alpha / (2 * np.pi)

f_qDP = avg_fracs[2]
f_hDP = avg_fracs[1]
mixing_sum = f_qDP + 0.7 * f_hDP
raw_delta = C * mixing_sum
delta_mu_e = raw_delta * S

print(f"Raw mixing boost: {raw_delta:.2e}")
print(f"Suppression S: {S:.4e}")
print(f"Electron δμ with lattice + FBS effects: {delta_mu_e:.2e}")

# Conservative upper bound
qDP_upper = f_qDP + 1.96 * std_fracs[2]
hDP_upper = f_hDP + 1.96 * std_fracs[1]
mixing_upper = qDP_upper + 0.7 * hDP_upper
raw_upper = C * mixing_upper
delta_upper = raw_upper * S
print(f"Conservative upper bound δμ_e: < {delta_upper:.2e}")